<a href="https://colab.research.google.com/github/pedro1ball/colab-notebooks_new/blob/main/task_linreg_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ Линейная регрессия

В данном задании мы рассмотрим набор данных об учащихся, собранный в 2006 году в одной из школ Португалии. Данные представлены в неудобном для машинного обучения виде, и содержат мусор. Ваша задача &mdash; привести их к надлежащему виду и обучить на них простую модель.

Данные состоят из четырех файлов:
- data.csv &mdash; основная таблица с информацией о учащихся
- scores.csv &mdash; список финальных оценок по одному из предметов (20-балльная шкала переведенная в проценты)
- attendance.csv &mdash; таблица посещений занятий по этому предмету
- school_support.txt &mdash; список учащихся, которым оказывается финансовая поддержка

Ваша задача &mdash; построить модель для предсказания финальных оценок исходя из всех остальных данных и проверить качество ее работы с помощью кросс-валидации. В качестве алгоритма мы будем использовать линейную регрессию

Расшифровка столбцов в data.csv для справки:
- age &mdash; возраст
- Medu &mdash; уровень образования матери (по некоторой условной шкале)
- Fedu &mdash; уровень образования отца (по некоторой условной шкале)
- traveltime &mdash; время в пути до школы (1 – < 15 мин., 2 – от 15 до 30 мин., 3 – от 30 мин. to 1 ч.
или 4 – > 1 ч.)
- studytime &mdash; время, затрачиваемое на занятия вне школы (1 – < 2 ч., 2 – от 2 до 5 ч., 3 – от 5 до 10 ч. или 4 – > 10 ч.)
- famrel &mdash; насколько хорошие отношения в семье у учащегося (по некоторой условной шкале)
- freetime &mdash; количество свободного времени вне школы (по некоторой условной шкале)
- goout &mdash; время, затрачиваемое на общение с друзьями (по некоторой условной шкале)
- Dalc &mdash; количество употребления алкоголя в учебные дни (по некоторой условной шкале)
- Walc &mdash; количество употребления алкоголя в неучебные дни (по некоторой условной шкале)
- health &mdash; уровень здоровья (по некоторой условной шкале)
- sex_M &mdash; пол: мужской (1) или женский (0)
- address_U &mdash; живет ли учащийся в городе (1) или в пригороде (0)
- famsize_LE3 &mdash; размер семьи: не больше 3 человек (1) или больше (0)
- Pstatus_T &mdash; живут ли родители вместе (1) или отдельно (0)
- nursery &mdash; посещал ли учащийся детский сад
- plans_university &mdash; планирует ли учащийся поступать в университет (-1 или 1)
- past_failures &mdash; количество неудовлетворительных оценок по другим предметам ранее (от 0 до 4)

*Примечание. Несколько признаков в данных содержат ошибки/проблемы/некорректности. Эти проблемы нужно исправить. Для
проверки &mdash; всего в данных таких проблем четыре.*

Начиная с 4 пункта делайте кроссвалидацию на 4 батча до и после, замеряйте результат

### Задача 1: сломанный признак (а может и не один)
__(1 балл)__

Загрузите таблицу data.csv.

Найдите в данных сломанный признак (он не соответствует описанию) и исправьте его.

In [6]:
import pandas as pd
import numpy as np

data = pd.read_csv('/content/data.csv')

print("Склеенный столбец и его значения:")
print(data['plans_universitypast_failures'].value_counts())

data['plans_university'] = np.where(data['plans_universitypast_failures'] > 0, 1, -1)
data['past_failures'] = data['plans_universitypast_failures'].abs() % 10

data.drop(columns=['plans_universitypast_failures'], inplace=True)

print("\nПроверка plans_university (-1 или 1):")
print(data['plans_university'].value_counts())

print("\nПроверка past_failures (от 0 до 4):")
print(data['past_failures'].value_counts())

Склеенный столбец и его значения:
plans_universitypast_failures
 10    513
 11     50
-10     36
-11     20
 12      9
 13      8
-12      7
-13      6
Name: count, dtype: int64

Проверка plans_university (-1 или 1):
plans_university
 1    580
-1     69
Name: count, dtype: int64

Проверка past_failures (от 0 до 4):
past_failures
0    549
1     70
2     16
3     14
Name: count, dtype: int64


### Задача 2: пропуски в данных
__(1 балл)__

Проверьте, есть ли в данных пропуски (значения NaN). Замените все пропущенные значения на среднее значение этого признака по столбцу.

*Hint: изучите в pandas функции loc, isnull, а также передачу булевых массивов в качестве индексов.*

In [19]:

missing_values = data.isnull().sum()
print("Пропуски по столбцам до обработки:")
print(missing_values[missing_values > 0])

for col in data.columns:
    if data[col].isnull().any():
        mean_val = data[col].mean()
        # data[col].isnull() выбирает строки с пропусками
        data.loc[data[col].isnull(), col] = mean_val

print("\nКоличество оставшихся NaN в датасете:", data.isnull().sum().sum())

Пропуски по столбцам до обработки:
Series([], dtype: int64)

Количество оставшихся NaN в датасете: 0


### Задача 3: нормализация данных
__(1 балл)__

Нормализуйте данные любым способом, в современных библиотеках нормализация часто встроена внутрь модели, поэтому тут разницы может и не быть в качестве, но сделать надо

In [20]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

data_normalized = pd.DataFrame(
    scaler.fit_transform(data),
    columns=data.columns,
    index=data.index
)

data = data_normalized

#Проверяем первые строки
data.head()
#должно быть от -2 до 2

,age,Medu,Fedu,traveltime,studytime,famrel,freetime,goout,Dalc,Walc,health,sex_M,address_U,famsize_LE3,Pstatus_T,nursery,plans_university,past_failures,school_support,attendance
0,-0.134205,1.310216,1.540715,-0.275347,0.083653,1.119748,0.780478,0.692772,-0.542692,-0.223769,1.012903,1.199937,0.660182,-0.648175,0.374963,0.495663,0.344914,-0.374305,-0.34211,-0.504730
1,-0.130277,1.310216,1.540715,-0.275347,-1.122808,1.119748,-0.171647,0.692772,-0.542692,-0.223769,1.012903,-0.833377,0.660182,-0.648175,0.374963,0.495663,0.344914,-0.374305,-0.34211,0.357863
2,-0.134205,-1.336039,-1.188832,0.093107,-1.122808,0.072606,1.732603,1.544607,0.547811,1.340184,1.012903,1.199937,-1.514733,1.542792,0.374963,0.495663,0.344914,-0.374305,-0.34211,0.789159
3,-0.126349,-1.336039,-0.278983,0.093107,-1.122808,-0.974536,0.780478,0.692772,0.547811,1.340184,0.320930,1.199937,0.660182,-0.648175,0.374963,-2.017502,-2.899275,-0.374305,-0.34211,-1.367323
4,-0.130277,-0.453954,-1.188832,0.093107,0.083653,0.072606,-1.123771,1.544607,-0.542692,-0.223769,1.012903,-0.833377,-1.514733,-0.648175,0.374963,0.495663,0.344914,-0.374305,-0.34211,0.789159


### Задача 4: кросс-валидация для исходных данных
__(1 балл)__

Загрузите файл scores.csv и протестируйте, как линейная регрессия предсказывает ответ сейчас (с помощью кросс-валидации).

Кроссвалидацию сделайте по 4 разбивкам. Выведите качество в каждом их разбиений.

*Hint: воспользуйтесь sklearn.linear_model и sklearn.model_selection.*

In [24]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

scores = pd.read_csv('/content/scores.csv', header=None)[0]

model = LinearRegression()

cv = KFold(n_splits=4, shuffle=True, random_state=73)

rmse_scores = -cross_val_score(model, data, scores, cv=4, scoring='neg_root_mean_squared_error')
for fold, score in enumerate(rmse_scores, start=1):
    print(f"Качество (RMSE) на фолде {fold}: {score:.4f}")

print(f"\nСреднее RMSE по кросс-валидации: {rmse_scores.mean():.4f}")

Качество (RMSE) на фолде 1: 12.8076
Качество (RMSE) на фолде 2: 15.4010
Качество (RMSE) на фолде 3: 15.8778
Качество (RMSE) на фолде 4: 15.1512

Среднее RMSE по кросс-валидации: 14.8094


### Задача 5: полные данные
__(2 балла)__

Воспользуйтесь файлами attendance.csv и school_support.txt для того, чтобы добавить новые признаки в данные. Желательно по максимуму использовать возможности pandas для упрощения преобразований.

school_suport число в строке значит что i-ый школьник из исходной таблицы получал мат помощь (обратите внимание что строк в файле меньше, подумайте как правильно импортировать данные)

Добавьте данные таким образом, чтобы качество выросло

In [17]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# материальную помощь
support_idx = pd.read_csv('/content/school_support.txt', header=None)[0]
data['school_support'] = 0
data.loc[support_idx, 'school_support'] = 1

# посещаемость
attendance = pd.read_csv('/content/attendance.csv', sep=';')
data['attendance'] = (attendance == '+').sum(axis=1)

#  обновленный датасет
scaler = StandardScaler()
data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns, index=data.index)

# Проверяем качествоcv=4
scores = pd.read_csv('/content/scores.csv', header=None)[0]
model = LinearRegression()

rmse_scores = -cross_val_score(model, data, scores, cv=4, scoring='neg_root_mean_squared_error')

for fold, score in enumerate(rmse_scores, start=1):
    print(f"Качество (RMSE) на фолде {fold}: {score:.4f}")

print(f"\nСреднее RMSE по кросс-валидации: {rmse_scores.mean():.4f}")

Качество (RMSE) на фолде 1: 12.8076
Качество (RMSE) на фолде 2: 15.4010
Качество (RMSE) на фолде 3: 15.8778
Качество (RMSE) на фолде 4: 15.1512

Среднее RMSE по кросс-валидации: 14.8094


### Задача 6: борьба с выбросами
__(1.5 балла)__

Качество предсказания может ухудшаться, если в данных присутствуют корректные значения признаков (с точки зрения чтения данных и применения методов), но не соответствующие реальным объектам. Например, данные могли быть введены в неверном формате, а потом слишком грубо приведены к общему виду, из-за чего ошибка не была замечена.
Попробуем от такого избавиться &mdash; а для этого такие объекты нужно сначала найти. Конечно, нам еще недоступны многие продвинутые способы, но давайте попробуем обойтись простыми.

Первый способ это сделать &mdash; посмотреть для каждого признака на распределение его значений и проверить крайние значения на правдоподобность. (постройте гистограммы для признаков, как минимум для подозрительных)

*Hint 1: используйте функцию DataFrame.hist*

*Hint 2: в описании датасета выше есть информация, необходимая для восстановления правильных значений*

In [ ]:
# Your code here
# ...

__(1.5 балла)__

Другой простой способ найти выбросы &mdash; сделать предсказание и посчитать ошибку на каждом объекте по отдельности и посмотреть на объекты с наибольшей ошибкой. Обучите линейную регрессию (функция fit) и для каждого объекта посчитайте среднеквадратичное отклонение. Постройте гистограмму распределения ошибок. Посмотрите на гистограмму и удалите из выборки те объекты на которых ошибка слишком большая.

Обратите внимание, что просто удалять все объекты с высокой ошибкой нельзя &mdash; это, конечно, хороший способ добиться меньшей ошибки (на данной выборке), но одновременно вы ухудшите обобщающую способность алгоритма. Вместо этого вам нужно найти однозначно ошибочные записи и их исправить.

*Hint: возможно, все проблемы уже были найдены первым способом; для проверки &mdash; в сумме здесь нужно исправить 3 проблемы.*

Для поиска ошибки на одном отдельном обьекте придётся обучить линейную регрессию руками. Частичный пример, допишите код. Постройте гистограмму распределения ошибок

In [ ]:
import sklearn
from sklearn import linear_model
regression = linear_model.LinearRegression().fit(data, result) #create model and train it
prediction = #calculate prediction for one object for vector x
error = (prediction - y)**2 #simple error - square error

SyntaxError: invalid syntax (3289234131.py, line 4)

In [ ]:
# Your code here
# ...

### Финальное предсказание и отчёт (1 балл)

Проведите предсказание еще раз и сравните качество с исходным. Запишите свои наблюдения - как изменялось качество обучения модели при использовании разных модификаций данных.

In [ ]:
# Your code here
# ...